# PlantScope v4.5 — Houseplant disease pipeline (training notebook)

Двухстадийный пайплайн распознавания болезней комнатных растений на T4 Colab.

## Целевая таксономия (9 классов)

| Класс | Семантика |
|-------|-----------|
| `gray_mold`              | Серая гниль (*Botrytis*) — пушистый серый налёт |
| `healthy`                | Здоровое растение |
| `leaf_spot`              | Пятнистости (грибковые/бактериальные) |
| `not_a_plant`            | Не растение (rejection class) |
| `nutrient_deficiency`    | Хлороз / дефицит питания |
| `overwatering`           | Переувлажнение (поникшие, мягкие листья) — **NEW v4.5** |
| `pest_insects`           | Вредители (тля/червец/щитовка/трипс/белокрылка) |
| `powdery_mildew`         | Мучнистая роса |
| `spider_mites`           | Паутинный клещ |

`pest_insects` объединяет 5 насекомых-вредителей. На разрешении 320px они
визуально неразличимы (~1-3мм на листе → ~5-10px после crop), и их лечение
одинаково (инсектицидное мыло / системные инсектициды).

`overwatering` (v4.5) — самая частая причина гибели комнатных растений.
Источник: Kaggle "Healthy and Wilted Houseplant Images" (~452 wilted фото).
Симптомы: мягкие, поникшие, желтеющие листья при влажной почве.

`gray_mold` (v4.4) — опциональный (зависит от того, наберёт ли iNat
*Botrytis cinerea* ≥60 фото). Если провалится — класс автоматически дропается
с явным warning. То же для других классов с graceful fallback.

## Что нового в v4.5

1. **Все iNat taxon ID реально проверены** через inaturalist.org/taxa/<id>.
   В v4.4 несколько ID были выдуманы — например, 47148 был Rosaceae (плоды/листья роз!),
   что наполнило бы gray_mold rose-фотками вместо грибов. Пофиксено.
2. **Класс `overwatering`** — главная причина гибели комнатных, до сих пор не было.
3. **Botrytis taxon фикс**: 118099 (несуществующий) → **324263** (реальный *B. cinerea*).
4. **Powdery mildew fallback**: добавлены реальные ID Erysiphaceae (55525) и
   Podosphaera (334961) на случай, если Erysiphales (55524) опять вернёт 0.

## Архитектура

| Стадия | Модель | Параметры | Что делает |
|--------|--------|-----------|------------|
| **Stage 1 — Detector** | YOLOv8n | ~3.0M | Локализация листьев (один класс `leaf`) |
| **Stage 2 — Classifier** | EfficientNet-B0 | ~5.3M | До 9 классов состояния + temperature |

## Бюджет

- T4 free Colab, ~3-3.5 ч end-to-end (cell 8 YOLO ~70 мин, cell 12 classifier ~70 мин)
- Disk peak: ~20 GB (raw data + YOLO disk-cache + crops). Свободно должно быть ≥30 GB.
- RAM: 12.7 GB лимит, держимся <8 GB благодаря `cache='disk'` + explicit cleanup в cell 12.

## Как использовать

1. Открыть в Google Colab → `Runtime → Change runtime type → T4 GPU`
2. (Один раз) загрузить `kaggle.json` для Kaggle API
3. (Опционально) разрешить mount Google Drive — артефакты будут автоматически
   сохраняться в `/content/drive/MyDrive/PlantScope_v45/` после каждой фазы,
   чтобы крэш не съел результаты
4. Run All сверху вниз
5. Артефакты в `/content/plantscope/out/` → переложить в `server/models/`:
   - `detector.pt`, `detector.onnx`
   - `classifier.pth`, `classifier.onnx`, `classifier_int8.onnx`
   - `classes.json`, `temperature.json`

## Совместимость с server/pipeline.py

`pipeline.py::_NON_DISEASE = {"healthy", "not_a_plant"}` — корректно работает с
новой таксономией: `pest_insects` и `gray_mold` обрабатываются как болезни
(triggers healthy fallback при низкой уверенности).


## 1. Установка зависимостей, проверка GPU, mount Drive

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'ultralytics>=8.2.0', 'kaggle',
    'onnx>=1.16', 'onnxruntime>=1.17', 'onnxsim>=0.4',
])

subprocess.run(
    ['apt-get', 'install', '-y', '-qq', 'aria2'],
    check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

import torch, torchvision, ultralytics, onnxruntime as ort
print(f'PyTorch:       {torch.__version__}')
print(f'TorchVision:   {torchvision.__version__}')
print(f'Ultralytics:   {ultralytics.__version__}')
print(f'ONNXRuntime:   {ort.__version__}')
print(f'CUDA:          {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:           {torch.cuda.get_device_name(0)}')
    print(f'VRAM:          {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:        {device}')

# ── Disk space check ──
disk_free = shutil.disk_usage('/content').free / 1e9
print(f'Disk free:     {disk_free:.1f} GB')
if disk_free < 25:
    print(f'⚠ Свободно только {disk_free:.1f} GB. Нужно ≥25 GB. Pipeline может упасть.')

# ── Drive mount (опционально, но рекомендуется) ──
# Автосейв артефактов в /content/drive/MyDrive/PlantScope_v45/
# работает даже после крэша runtime, если Drive смонтирован.
DRIVE_DIR = None
try:
    from google.colab import drive  # type: ignore
    if not Path('/content/drive/MyDrive').exists():
        print('\nMounting Google Drive (откроется окно авторизации)...')
        drive.mount('/content/drive', force_remount=False)
    if Path('/content/drive/MyDrive').exists():
        DRIVE_DIR = Path('/content/drive/MyDrive/PlantScope_v45')
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        print(f'Drive backup:  {DRIVE_DIR}')
    else:
        print('\nDrive не смонтирован. Артефакты будут только в /content/.')
except Exception as e:
    print(f'\nDrive mount пропущен ({e}). Артефакты будут только в /content/.')


def save_to_drive(local_path: Path, label: str = '') -> None:
    """Best-effort copy в Drive backup. Молча пропускает если Drive нет."""
    if DRIVE_DIR is None or not local_path.exists():
        return
    try:
        shutil.copy2(local_path, DRIVE_DIR / local_path.name)
        if label:
            print(f'  ↑ Drive: {label} ({local_path.name})')
    except Exception as e:
        print(f'  ⚠ Drive save failed для {local_path.name}: {e}')


## 2. Загружаем датасеты

Все источники свободны для исследовательского использования. Для коммерческого
деплоя обратите внимание: Wilted Houseplants — Google scrape, лицензии mixed.

| Источник | Лицензия | Что берём | Зачем |
|----------|----------|-----------|-------|
| PlantDoc OD | CC-BY 4.0 | bbox-аннотации листьев | детектор |
| PlantVillage (sparse) | CC-BY-3.0 | spider_mites, powdery_mildew, leaf_spot, TYLCV (хлороз) | классификатор (clean) |
| iNaturalist Open Data | CC0/CC-BY/CC-BY-SA | Pseudococcidae/Coccoidea/Aleyrodidae/Aphidoidea/Thysanoptera/Tetranychus/Erysiphales/*Botrytis cinerea* | классификатор (wild) |
| Houseplant Species (Kaggle) | CC0 | комнатные растения | healthy после авто-кропа |
| **Wilted Houseplants (Kaggle)** | mixed | wilted houseplants | **overwatering** (новое в v4.5) |
| COCO val2017 + ann | CC-BY 4.0 | случайные «не-растения» | not_a_plant |

iNaturalist API ограничен 60 req/min. Скрипт уважает rate-limit и кеширует загрузки в `raw/inaturalist/<taxon>/`.

**Все iNat taxon ID проверены вручную через inaturalist.org/taxa/<id> в апреле 2026.**
В v4.4 несколько ID были выдуманы (например, 47148 был Rosaceae вместо рода Botrytis,
что наполнило бы gray_mold rose-фотками). v4.5 — все ID реальные.

**Botrytis (gray_mold)** — failover-цепочка вид → род → семейство:
*B. cinerea* (324263) → *Botrytis* genus (324264) → *Sclerotiniaceae* (204631).
Если все три вернут <60 фото — класс автоматически дропается (graceful fallback).


In [ ]:
import subprocess, time, json, re, gc
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import requests
from requests.adapters import HTTPAdapter

WORKDIR = Path('/content/plantscope')
RAW = WORKDIR / 'raw'
RAW.mkdir(parents=True, exist_ok=True)


def _has(cmd: str) -> bool:
    return subprocess.run(['which', cmd], stdout=subprocess.DEVNULL,
                          stderr=subprocess.DEVNULL).returncode == 0


def _link_or_copy(src: Path, dst: Path) -> None:
    """Hardlink с откатом на copy2 при cross-fs."""
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except (OSError, FileNotFoundError):
        shutil.copy2(src, dst)


def _download_zip(url: str, out: Path) -> None:
    if _has('aria2c'):
        subprocess.check_call([
            'aria2c', '-x', '16', '-s', '16', '-q',
            '--allow-overwrite=true', '--auto-file-renaming=false',
            '-d', str(out.parent), '-o', out.name, url,
        ])
    else:
        subprocess.check_call(['wget', '-q', '-O', str(out), url])


def _unzip(src: Path, dst: Path, only: str | None = None) -> None:
    """Системный unzip быстрее zipfile на больших архивах.

    Порядок аргументов unzip: [options] zipfile [files-to-extract] [-d dest].
    `only` идёт ПОСЛЕ архива, не до — иначе unzip ищет архив по `only` и падает.
    """
    cmd = ['unzip', '-q', '-o', str(src)]
    if only:
        cmd.append(only)
    cmd.extend(['-d', str(dst)])
    subprocess.check_call(cmd)


# ── 2.1 PlantDoc Object Detection ────────────────────────────────
plantdoc_dir = RAW / 'plantdoc_od'

def fetch_plantdoc():
    if plantdoc_dir.exists():
        return f'PlantDoc уже на диске ({sum(1 for _ in plantdoc_dir.rglob("*.jpg"))} jpg).'
    subprocess.check_call([
        'git', 'clone', '--depth', '1', '-q',
        'https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset.git',
        str(plantdoc_dir),
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return f'PlantDoc OD: {sum(1 for _ in plantdoc_dir.rglob("*.jpg"))} jpg.'


# ── 2.2 PlantVillage sparse-checkout ─────────────────────────────
PV_FOLDERS = {
    'spider_mites': [
        'raw/color/Tomato___Spider_mites Two-spotted_spider_mite',
    ],
    'powdery_mildew': [
        'raw/color/Squash___Powdery_mildew',
        'raw/color/Cherry_(including_sour)___Powdery_mildew',
    ],
    'leaf_spot': [
        'raw/color/Tomato___Septoria_leaf_spot',
        'raw/color/Tomato___Bacterial_spot',
        'raw/color/Pepper,_bell___Bacterial_spot',
    ],
    # TYLCV визуально проявляется как межжилковый хлороз — те же симптомы,
    # что и при нехватке железа/азота на комнатных.
    'nutrient_deficiency': [
        'raw/color/Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    ],
}
pv_root = RAW / 'plantvillage_repo'
pv_extracted = RAW / 'plantvillage_extracted'

def fetch_plantvillage():
    if pv_extracted.exists() and any(pv_extracted.iterdir()):
        return 'PlantVillage уже распакован.'
    pv_extracted.mkdir(exist_ok=True)
    if not pv_root.exists():
        subprocess.check_call([
            'git', 'clone', '--filter=blob:none', '--no-checkout', '--depth', '1', '-q',
            'https://github.com/spMohanty/PlantVillage-Dataset.git', str(pv_root),
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call(
        ['git', '-C', str(pv_root), 'sparse-checkout', 'init', '--cone'],
        stdout=subprocess.DEVNULL,
    )
    all_paths = [p for paths in PV_FOLDERS.values() for p in paths]
    subprocess.check_call(
        ['git', '-C', str(pv_root), 'sparse-checkout', 'set'] + all_paths,
        stdout=subprocess.DEVNULL,
    )
    subprocess.check_call(
        ['git', '-C', str(pv_root), 'checkout', '-q'],
        stdout=subprocess.DEVNULL,
    )
    total = 0
    for cls, paths in PV_FOLDERS.items():
        out = pv_extracted / cls
        out.mkdir(exist_ok=True)
        for p in paths:
            src = pv_root / p
            if not src.exists():
                continue
            for f in src.iterdir():
                if f.suffix.lower() in ('.jpg', '.jpeg', '.png'):
                    _link_or_copy(f, out / f.name)
                    total += 1
    return f'PlantVillage: {total} файлов слинковано.'


# ── 2.3 COCO val2017 ─────────────────────────────────────────────
coco_dir = RAW / 'coco_val2017'

def fetch_coco_images():
    if coco_dir.exists() and any(coco_dir.glob('*.jpg')):
        return f'COCO val2017 уже на диске ({sum(1 for _ in coco_dir.glob("*.jpg"))} jpg).'
    coco_zip = RAW / 'coco_val2017.zip'
    _download_zip('http://images.cocodataset.org/zips/val2017.zip', coco_zip)
    _unzip(coco_zip, RAW)
    extracted = RAW / 'val2017'
    if extracted.exists():
        extracted.rename(coco_dir)
    coco_zip.unlink(missing_ok=True)
    return f'COCO val2017: {sum(1 for _ in coco_dir.glob("*.jpg"))} jpg распаковано.'


coco_ann = RAW / 'annotations' / 'instances_val2017.json'

def fetch_coco_annotations():
    if coco_ann.exists():
        return 'COCO annotations уже на диске.'
    ann_zip = RAW / 'coco_ann.zip'
    _download_zip(
        'http://images.cocodataset.org/annotations/annotations_trainval2017.zip',
        ann_zip,
    )
    _unzip(ann_zip, RAW, only='annotations/instances_val2017.json')
    ann_zip.unlink(missing_ok=True)
    return 'COCO annotations распакованы.'


# ── 2.4 iNaturalist Open Data ────────────────────────────────────
# API-листинг последовательный (rate-limit ≤60 req/min), скачивание
# с CDN распараллелено на 16 потоков.

# Каждый ключ → список fallback-таксонов (пробуем по очереди до набора TARGET).
# Все taxon ID сверены вручную с inaturalist.org/taxa/<id> (April 2026).
# v4.5: КРИТИЧЕСКИЙ ФИКС - таксоны в v4.4 были частично выдуманы:
#   118099 не существовал (правильный B. cinerea = 324263)
#   47148 = Rosaceae (рожи!) — наполнил бы gray_mold rose-фотками
#   124319/124320/124342 — несуществующие или неверные ID
#   53121 — несуществующий ID
# Здесь все ID реальные, проверены через web-search inaturalist.org/taxa/<N>.
INAT_TAXA: dict[str, dict] = {
    'mealybug':         {'fallback': [(49574, 'Pseudococcidae')]},  # ✓
    'scale_insect':     {'fallback': [(60922, 'Coccoidea (superfamily)')]},  # ✓
    'spider_mite_wild': {'fallback': [(203333, 'Tetranychus urticae'),  # ✓
                                      (128552, 'Tetranychidae (family)')]},  # ✓ (was 53121)
    # Erysiphales в v4.3 fetch вернул 0 — теперь fallback на семейство и род.
    'powdery_mildew':   {'fallback': [(55524,  'Erysiphales (order)'),    # ✓
                                      (55525,  'Erysiphaceae (family)'),  # ✓ (was 124319)
                                      (334961, 'Podosphaera (genus)')]},  # ✓ (was 124320)
    'aphids':           {'fallback': [(52380, 'Aphidoidea')]},  # ✓
    'thrips':           {'fallback': [(83201, 'Thysanoptera')]},  # ✓
    'whitefly':         {'fallback': [(154784, 'Aleyrodidae')]},  # ✓
    # gray_mold: цепочка вид → род → семейство.
    'gray_mold':        {'fallback': [(324263, 'Botrytis cinerea'),  # ✓ (was 118099 — fake)
                                      (324264, 'Botrytis (genus)'),  # ✓ (was 47148 — Rosaceae!)
                                      (204631, 'Sclerotiniaceae (family)')]},  # ✓ (was 151869)
}
TARGET_PER_TAXON = 600
INAT_BASE = 'https://api.inaturalist.org/v1/observations'

_HTTP = requests.Session()
_HTTP.mount('https://', HTTPAdapter(pool_connections=20, pool_maxsize=20))
_HTTP.mount('http://',  HTTPAdapter(pool_connections=20, pool_maxsize=20))

# Magic bytes для проверки, что скачался реальный image (не HTML-ошибка).
_IMG_MAGIC = (b'\xff\xd8\xff', b'\x89PNG\r\n\x1a\n', b'GIF87a', b'GIF89a', b'RIFF')

def _looks_like_image(data: bytes) -> bool:
    return any(data.startswith(m) for m in _IMG_MAGIC)


def _inat_download_one(url: str, dst: Path) -> bool:
    if dst.exists():
        return True
    try:
        r = _HTTP.get(url, timeout=15)
    except Exception:
        return False
    if r.status_code != 200 or len(r.content) <= 4096:
        return False
    if not _looks_like_image(r.content):
        return False
    try:
        dst.write_bytes(r.content)
        return True
    except Exception:
        return False


def _inat_fetch_taxon(taxon_id: int, target: int, out_dir: Path,
                     already_have: int) -> int:
    """Скачивает фото одного таксона. Возвращает количество добавленных."""
    page = 1
    saved = already_have
    with ThreadPoolExecutor(max_workers=16) as pool:
        while saved < target and page <= 30:
            q = {
                'taxon_id': taxon_id,
                'photo_license': 'cc0,cc-by,cc-by-sa',
                'quality_grade': 'research',
                'identified': 'true',
                'per_page': 200,
                'page': page,
            }
            try:
                r = _HTTP.get(INAT_BASE, params=q, timeout=30)
                if r.status_code != 200:
                    print(f'    iNat HTTP {r.status_code} taxon={taxon_id} page={page}, stopping')
                    break
                data = r.json()
            except Exception as e:
                print(f'    iNat error taxon={taxon_id} page={page}: {e}'); break
            results = data.get('results', [])
            if not results:
                break
            jobs = []
            for obs in results:
                for ph in obs.get('photos', []):
                    lic = (ph.get('license_code') or '').lower()
                    if lic not in ('cc0', 'cc-by', 'cc-by-sa'):
                        continue
                    url = (ph.get('url') or '') \
                        .replace('/square.', '/medium.') \
                        .replace('/small.', '/medium.')
                    if not url:
                        continue
                    ext = '.jpg'
                    m = re.search(r'\.(jpe?g|png)(?:\?|$)', url, re.IGNORECASE)
                    if m:
                        ext = '.' + m.group(1).lower()
                    fn = out_dir / f"t{taxon_id}_{obs['id']}_{ph['id']}{ext}"
                    jobs.append((url, fn))
                    if len(jobs) + saved >= target * 2:
                        break
                if len(jobs) + saved >= target * 2:
                    break
            futures = [pool.submit(_inat_download_one, url, fn) for url, fn in jobs]
            for fut in as_completed(futures):
                if fut.result():
                    saved += 1
                if saved >= target:
                    break
            page += 1
            time.sleep(1.1)  # ≤60 req/min на API
    return saved


def _inat_fetch_with_fallback(key: str, fallbacks: list, target: int,
                              out_dir: Path) -> tuple[int, str]:
    """Перебирает fallback-таксоны до набора target фото. Возвращает (count, source_label)."""
    out_dir.mkdir(parents=True, exist_ok=True)
    have = sum(1 for f in out_dir.iterdir()
               if f.suffix.lower() in ('.jpg', '.jpeg', '.png'))
    if have >= target:
        return have, 'cached'

    sources = []
    saved = have
    for tid, label in fallbacks:
        before = saved
        saved = _inat_fetch_taxon(tid, target, out_dir, saved)
        gained = saved - before
        if gained > 0:
            sources.append(f'{label}+{gained}')
        if saved >= target:
            break
    return saved, ','.join(sources) if sources else 'none'


inat_root = RAW / 'inaturalist'

def fetch_inaturalist():
    inat_root.mkdir(exist_ok=True)
    msgs: list[str] = []
    for key, cfg in INAT_TAXA.items():
        n, src = _inat_fetch_with_fallback(
            key, cfg['fallback'], TARGET_PER_TAXON, inat_root / key,
        )
        msgs.append(f'iNat {key}: {n}  [{src}]')
    return '\n  '.join(msgs)


# ── Параллельный fetch ────────────────────────────────────────────
JOBS = [
    ('plantdoc',     fetch_plantdoc),
    ('plantvillage', fetch_plantvillage),
    ('coco_images',  fetch_coco_images),
    ('coco_ann',     fetch_coco_annotations),
    ('inaturalist',  fetch_inaturalist),
]
print(f'Параллельно качаем {len(JOBS)} источника(ов)…  '
      f'(aria2c={"yes" if _has("aria2c") else "no, wget fallback"})')
t0 = time.time()
with ThreadPoolExecutor(max_workers=len(JOBS)) as pool:
    futures = {pool.submit(fn): name for name, fn in JOBS}
    for fut in as_completed(futures):
        name = futures[fut]
        try:
            print(f'  ✓ {name}: {fut.result()}')
        except Exception as e:
            print(f'  ✗ {name}: {e}')
print(f'\nDownload total: {time.time() - t0:.1f}s')


# ── 2.5 Houseplant Species (Kaggle) ──────────────────────────────
hp_dir = RAW / 'houseplant'
if not hp_dir.exists():
    kj = Path.home() / '.kaggle' / 'kaggle.json'
    if not kj.exists():
        try:
            from google.colab import files  # type: ignore
            print('Загрузите kaggle.json (Kaggle → Account → Create API Token)…')
            up = files.upload()
            kj.parent.mkdir(parents=True, exist_ok=True)
            for n, d in up.items():
                kj.write_bytes(d)
            kj.chmod(0o600)
        except Exception as e:
            print(f'kaggle.json не получен: {e}')
    if kj.exists():
        try:
            subprocess.check_call(['kaggle', 'datasets', 'download', '-d',
                'kacpergregorowicz/house-plant-species', '-p', str(hp_dir),
                '--unzip', '-q'])
        except Exception as e:
            print(f'Kaggle Houseplant Species: {e}')
n_hp = sum(1 for _ in hp_dir.rglob('*.jpg')) + sum(1 for _ in hp_dir.rglob('*.jpeg')) \
       if hp_dir.exists() else 0
print(f'Houseplant Species: {n_hp} images')

# ── 2.6 Wilted Houseplants (Kaggle) для overwatering ─────────────
# v4.5 NEW: набор 904 фото "healthy"/"wilted" комнатных. Wilted-папка
# даёт визуальную сигнатуру overwatering (главная причина — переувлажнение).
# Источник: Russell Chan, 2020. Изображения собраны через Google Images,
# поэтому исходные лицензии mixed; для исследовательской/образовательной
# цели приемлемо. Для коммерческого деплоя нужна замена на собственный
# датасет с проверенными лицензиями.
wilted_dir = RAW / 'wilted_houseplants'
if not wilted_dir.exists():
    kj = Path.home() / '.kaggle' / 'kaggle.json'
    if kj.exists():
        try:
            subprocess.check_call(['kaggle', 'datasets', 'download', '-d',
                'russellchan/healthy-and-wilted-houseplant-images',
                '-p', str(wilted_dir), '--unzip', '-q'])
        except Exception as e:
            print(f'Kaggle Wilted Houseplants: {e}')

# Структура датасета: house_plant_species/Healthy/ и house_plant_species/Wilted/
# или варианты — попробуем найти "wilted" подпапку любого регистра.
wilted_imgs_path = None
if wilted_dir.exists():
    for p in wilted_dir.rglob('*'):
        if p.is_dir() and p.name.lower() in ('wilted', 'unhealthy'):
            wilted_imgs_path = p
            break
n_wilted = sum(1 for _ in wilted_imgs_path.rglob('*.jpg')) + \
           sum(1 for _ in wilted_imgs_path.rglob('*.jpeg')) + \
           sum(1 for _ in wilted_imgs_path.rglob('*.png')) \
           if wilted_imgs_path else 0
print(f'Wilted Houseplants: {n_wilted} images (overwatering source)')

print('\nДанные готовы. Источники:')
print(f'  PlantDoc OD     : {plantdoc_dir}')
print(f'  PlantVillage    : {pv_extracted}')
print(f'  COCO val2017    : {coco_dir}')
print(f'  Houseplant      : {hp_dir}')
print(f'  Wilted plants   : {wilted_imgs_path}')
print(f'  iNaturalist     : {inat_root}')

# Disk usage report
disk_used = shutil.disk_usage('/content').used / 1e9
disk_free = shutil.disk_usage('/content').free / 1e9
print(f'\nDisk: used {disk_used:.1f} GB, free {disk_free:.1f} GB')


## 3. YOLO-датасет: один класс `leaf`

Детектор одноклассовый: только локализует листья, решение «болеет ли» делает классификатор.
PlantDoc даёт PASCAL VOC XML на 27 классов — все мапятся в `leaf`. Split 85/15 train/val.


In [ ]:
import xml.etree.ElementTree as ET
import random

YOLO_DIR = WORKDIR / 'detector'
if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)
for sub in ('images/train', 'images/val', 'labels/train', 'labels/val'):
    (YOLO_DIR / sub).mkdir(parents=True, exist_ok=True)

random.seed(42)
pairs: list[tuple[Path, Path]] = []
for split_name in ('TRAIN', 'TEST'):
    split_dir = plantdoc_dir / split_name
    if not split_dir.exists():
        continue
    for xml in split_dir.rglob('*.xml'):
        for ext in ('.jpg', '.JPG', '.jpeg', '.png'):
            img = xml.with_suffix(ext)
            if img.exists():
                pairs.append((img, xml))
                break
print(f'PlantDoc XML pairs: {len(pairs)}')
random.shuffle(pairs)
cut = int(len(pairs) * 0.85)
train_pairs, val_pairs = pairs[:cut], pairs[cut:]


def write_yolo(pair_list, split):
    kept = 0
    for img, xml in pair_list:
        try:
            root = ET.parse(xml).getroot()
        except ET.ParseError:
            continue
        size = root.find('size')
        if size is None:
            continue
        W = float(size.findtext('width') or 0)
        H = float(size.findtext('height') or 0)
        if W <= 0 or H <= 0:
            continue
        lines = []
        for obj in root.findall('object'):
            name = (obj.findtext('name') or '').lower()
            if 'leaf' not in name:
                continue
            bb = obj.find('bndbox')
            if bb is None:
                continue
            xmin = float(bb.findtext('xmin') or 0)
            ymin = float(bb.findtext('ymin') or 0)
            xmax = float(bb.findtext('xmax') or 0)
            ymax = float(bb.findtext('ymax') or 0)
            if xmax <= xmin or ymax <= ymin:
                continue
            cx = ((xmin + xmax) / 2) / W
            cy = ((ymin + ymax) / 2) / H
            bw = (xmax - xmin) / W
            bh = (ymax - ymin) / H
            cx, cy = min(max(cx, 0), 1), min(max(cy, 0), 1)
            bw, bh = min(max(bw, 0.001), 1), min(max(bh, 0.001), 1)
            lines.append(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
        if not lines:
            continue
        _link_or_copy(img, YOLO_DIR / f'images/{split}' / img.name)
        (YOLO_DIR / f'labels/{split}' / (img.stem + '.txt')).write_text(
            '\n'.join(lines), encoding='utf-8')
        kept += 1
    return kept


n_tr = write_yolo(train_pairs, 'train')
n_vl = write_yolo(val_pairs, 'val')
print(f'YOLO train={n_tr}  val={n_vl}')

data_yaml = YOLO_DIR / 'data.yaml'
data_yaml.write_text(
    f'path: {YOLO_DIR}\ntrain: images/train\nval: images/val\nnc: 1\nnames: [leaf]\n',
    encoding='utf-8')
print('data.yaml →', data_yaml)


## 4. Тренируем YOLOv8n (Stage 1)

Одноклассовый детектор сходится быстрее. T4: ~70 мин на 60 эпох.
Ожидаемые финальные метрики: mAP50 ≈ 0.74-0.78, mAP50-95 ≈ 0.50.

- `cache='disk'` — экономит RAM Colab (~9.6 GB на cache vs RAM)
- `mosaic=1.0` + `close_mosaic=10` — устойчивость к шумным фото; финальные эпохи без mosaic
- `hsv_*` — устойчивость к индоор-освещению (тёплая лампа, окно)
- AMP + cos_lr + patience=12 — стандартный recipe

Артефакты сразу копируются на Drive (если смонтирован).


In [ ]:
from ultralytics import YOLO

det_model = YOLO('yolov8n.pt')

det_results = det_model.train(
    data=str(data_yaml),
    epochs=60,
    imgsz=640,
    batch=32,
    patience=12,
    cos_lr=True,
    amp=True,
    cache='disk',
    workers=4,
    save_period=-1,
    close_mosaic=10,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    project=str(WORKDIR / 'yolo_runs'),
    name='detector',
    exist_ok=True,
    verbose=True,
)

best_pt = WORKDIR / 'yolo_runs' / 'detector' / 'weights' / 'best.pt'
assert best_pt.exists(), f'YOLO best.pt не сохранён в {best_pt}'
MODELS_OUT = WORKDIR / 'out'
MODELS_OUT.mkdir(exist_ok=True)
shutil.copy2(best_pt, MODELS_OUT / 'detector.pt')
save_to_drive(MODELS_OUT / 'detector.pt', 'detector PT after YOLO train')

det_model.export(format='onnx', imgsz=640, opset=13, simplify=True, dynamic=False)
onnx_src = best_pt.with_suffix('.onnx')
if onnx_src.exists():
    shutil.copy2(onnx_src, MODELS_OUT / 'detector.onnx')
    save_to_drive(MODELS_OUT / 'detector.onnx', 'detector ONNX')
    print('Detector ONNX →', MODELS_OUT / 'detector.onnx')

metrics = det_model.val(data=str(data_yaml), imgsz=640, plots=False, verbose=False)
print(f'mAP50: {metrics.box.map50:.3f}   mAP50-95: {metrics.box.map:.3f}')


## 5. Классификаторный датасет (Stage 2)

**Целевая таксономия v4.5 (до 9 классов с graceful fallback):**

| Класс | Источник | План количества |
|-------|----------|----------------|
| `gray_mold`              | iNat *Botrytis cinerea* (324263) → род → семейство | 200-600 |
| `healthy`                | Houseplant Species через детектор (multi-crop+NMS) | 2500 |
| `leaf_spot`              | PlantVillage Septoria/Bacterial spot | 500 |
| `not_a_plant`            | COCO val2017 минус plant-like | 1200 |
| `nutrient_deficiency`    | PlantVillage TYLCV (хлороз = нехватка N/Fe) | 500 |
| `overwatering`           | **Kaggle Wilted Houseplants (NEW v4.5)** | ~452 |
| `pest_insects`           | **5 источников iNat объединены** + PV spider_mites | 2000-3000 |
| `powdery_mildew`         | PV Squash/Cherry + iNat Erysiphales (failover к Erysiphaceae/Podosphaera) | 800-1100 |
| `spider_mites`           | PV Tomato Spider Mites + iNat *Tetranychus urticae* | 800 |

`pest_insects` — объединение mealybug+scale_insect+aphids+thrips+whitefly. Это
~5 × 480 = 2400 фото cap. На 320px они визуально неразличимы (~1-3мм на листе),
а лечение одинаковое — объединение даёт устойчивый class boundary.

`overwatering` — wilted-фото из Kaggle. Главные визуальные признаки: мягкие,
поникшие, желтеющие листья при влажной почве. Самая частая причина гибели
комнатных растений.

**Стратифицированный split** (15% val, мин. 8 примеров) — каждый класс получает
ровно ту долю val, иначе редкие классы дают шумную метрику.

**Multi-crop healthy с NMS (IoU<0.5)** — гарантирует кропы из разных листьев,
не пересекающиеся bbox на одном.

В конце детектор YOLO **явно удаляется** из VRAM перед classifier-фазой.


In [ ]:
import json
from PIL import Image
import numpy as np
import torch
from ultralytics import YOLO

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MODELS_OUT = WORKDIR / 'out'
MODELS_OUT.mkdir(exist_ok=True)

CLS_DIR = WORKDIR / 'classifier'
if CLS_DIR.exists():
    shutil.rmtree(CLS_DIR)

# Алфавитный порядок (как ImageFolder.classes отсортирует).
TARGET_CLASSES = [
    'gray_mold',
    'healthy',
    'leaf_spot',
    'not_a_plant',
    'nutrient_deficiency',
    'overwatering',  # v4.5 NEW: wilted houseplants from Kaggle
    'pest_insects',
    'powdery_mildew',
    'spider_mites',
]
random.seed(42)
VAL_RATIO = 0.15
MIN_VAL_PER_CLASS = 8

for split in ('train', 'val'):
    for cls in TARGET_CLASSES:
        (CLS_DIR / split / cls).mkdir(parents=True, exist_ok=True)

# Стратифицированный split: собираем все источники по классам, потом
# случайно делим каждый класс с фиксированной долей val.
# Каждый элемент: (kind, payload) где kind ∈ {'file', 'pil'}.
pool: dict[str, list[tuple]] = {c: [] for c in TARGET_CLASSES}


# ── 5.1 PlantVillage кропы ──
PV_TO_CLASS = {
    'spider_mites': 'spider_mites',
    'powdery_mildew': 'powdery_mildew',
    'leaf_spot': 'leaf_spot',
    'nutrient_deficiency': 'nutrient_deficiency',
}
PV_CAP = {
    'spider_mites': 400,
    'powdery_mildew': 500,
    'leaf_spot': 500,
    'nutrient_deficiency': 500,
}
for pv_cls, dst_cls in PV_TO_CLASS.items():
    src_dir = pv_extracted / pv_cls
    if not src_dir.exists():
        continue
    imgs = [p for p in src_dir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    random.shuffle(imgs)
    selected = imgs[:PV_CAP[pv_cls]]
    for img in selected:
        pool[dst_cls].append(('file', img))
    print(f'PV {pv_cls} → {dst_cls}: {len(selected)}')

# ── 5.2 iNaturalist → объединение в pest_insects + остальные ──
# v4.4 KEY DECISION: 5 классов насекомых сливаются в pest_insects.
# По 480 фото с источника = до 2400 в классе (cap). Это намеренно
# крупнее остальных — насекомые на 320px разрешении внутренне разнообразны
# (тля != щитовка визуально), нужно больше примеров для устойчивого decision boundary.
INAT_TO_PEST = ['mealybug', 'scale_insect', 'aphids', 'thrips', 'whitefly']
INAT_PEST_PER_SOURCE = 480  # ~2400 итого до cap
PEST_CAP = 2400

INAT_OTHER = {
    'spider_mite_wild': ('spider_mites', 600),
    'powdery_mildew':   ('powdery_mildew', 600),
    'gray_mold':        ('gray_mold', 600),
}

for key in INAT_TO_PEST:
    src_dir = inat_root / key
    if not src_dir.exists():
        continue
    imgs = [p for p in src_dir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    random.shuffle(imgs)
    selected = imgs[:INAT_PEST_PER_SOURCE]
    for img in selected:
        pool['pest_insects'].append(('file', img))
    print(f'iNat {key} → pest_insects: {len(selected)}')

# Cap pest_insects (на случай если все 5 источников по 600+ дали >2400)
if len(pool['pest_insects']) > PEST_CAP:
    random.shuffle(pool['pest_insects'])
    pool['pest_insects'] = pool['pest_insects'][:PEST_CAP]
print(f'pest_insects total after cap: {len(pool["pest_insects"])}')

for key, (dst_cls, cap) in INAT_OTHER.items():
    src_dir = inat_root / key
    if not src_dir.exists():
        continue
    imgs = [p for p in src_dir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    random.shuffle(imgs)
    selected = imgs[:cap]
    for img in selected:
        pool[dst_cls].append(('file', img))
    print(f'iNat {key} → {dst_cls}: {len(selected)}')


# ── 5.3 Houseplant Species → healthy: BATCHED multi-crop + NMS ──
HP_TARGET_HEALTHY = 2500
CROPS_PER_PHOTO = 3
DETECT_BATCH = 16


def _iou(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    a_area = (ax2 - ax1) * (ay2 - ay1)
    b_area = (bx2 - bx1) * (by2 - by1)
    return inter / (a_area + b_area - inter + 1e-9)


def _nms_keep(boxes, max_keep, iou_thr=0.5):
    boxes_sorted = sorted(boxes, key=lambda b: -((b[2]-b[0])*(b[3]-b[1])))
    kept = []
    for b in boxes_sorted:
        if any(_iou(b, k) > iou_thr for k in kept):
            continue
        kept.append(b)
        if len(kept) >= max_keep:
            break
    return kept


if hp_dir.exists():
    det = YOLO(str(MODELS_OUT / 'detector.pt'))
    hp_imgs = []
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG'):
        hp_imgs.extend(hp_dir.rglob(ext))
    random.shuffle(hp_imgs)
    saved_crop = 0
    healthy_added = 0

    def _open_safe(p):
        try:
            return Image.open(p).convert('RGB')
        except Exception:
            return None

    i = 0
    while i < len(hp_imgs) and healthy_added < HP_TARGET_HEALTHY:
        chunk = hp_imgs[i:i + DETECT_BATCH]
        i += DETECT_BATCH
        pils = [_open_safe(p) for p in chunk]
        valid_pairs = [(p, pi) for p, pi in zip(chunk, pils) if pi is not None]
        if not valid_pairs:
            continue
        arrs = [np.array(pi) for _, pi in valid_pairs]
        try:
            results = det.predict(arrs, imgsz=640, conf=0.4,
                                  verbose=False, device=device)
        except Exception:
            continue
        for (path, pil), res in zip(valid_pairs, results):
            if healthy_added >= HP_TARGET_HEALTHY:
                break
            if res.boxes is None or len(res.boxes) == 0:
                continue
            boxes = res.boxes.xyxy.cpu().numpy().tolist()
            H, W = pil.size[1], pil.size[0]
            kept = _nms_keep(boxes, CROPS_PER_PHOTO, iou_thr=0.5)
            for b in kept:
                x1, y1, x2, y2 = (int(round(v)) for v in b)
                pad_x = int((x2 - x1) * 0.1)
                pad_y = int((y2 - y1) * 0.1)
                x1, y1 = max(0, x1 - pad_x), max(0, y1 - pad_y)
                x2, y2 = min(W, x2 + pad_x), min(H, y2 + pad_y)
                if x2 - x1 < 32 or y2 - y1 < 32:
                    continue
                crop = pil.crop((x1, y1, x2, y2))
                pool['healthy'].append(('pil', crop))
                saved_crop += 1
                healthy_added += 1
                if healthy_added >= HP_TARGET_HEALTHY:
                    break
    print(f'Houseplant Species → healthy: {saved_crop} crops via detector (batched multi-crop)')

    # КРИТИЧНО: освобождаем VRAM от YOLO до старта classifier-фазы.
    # Без этого тренировочный батч classifier'а упирается в OOM.
    del det
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

# ── 5.4 COCO → not_a_plant ──
PLANT_LIKE_COCO_IDS = {
    64,  # potted plant
    86,  # vase
    56,  # broccoli
    50, 51, 52, 53, 54, 55, 57, 58, 59, 60, 61,
}
banned: set[int] = set()
if coco_ann.exists():
    coco_meta = json.loads(coco_ann.read_text(encoding='utf-8'))
    for a in coco_meta.get('annotations', []):
        if a.get('category_id') in PLANT_LIKE_COCO_IDS:
            banned.add(a['image_id'])
    print(f'COCO ban-list: {len(banned)} images excluded')
    del coco_meta
    gc.collect()


def _cid(p: Path) -> int:
    try:
        return int(p.stem)
    except ValueError:
        return -1


coco_imgs = [p for p in coco_dir.glob('*.jpg') if _cid(p) not in banned]
random.shuffle(coco_imgs)
COCO_CAP = 1200
for img in coco_imgs[:COCO_CAP]:
    pool['not_a_plant'].append(('file', img))
print(f'COCO → not_a_plant: {min(len(coco_imgs), COCO_CAP)}')

# ── 5.5 Wilted Houseplants → overwatering (v4.5 NEW) ──
# Wilting — главный визуальный симптом overwatering на комнатных
# (мягкие, поникшие, желтеющие листья при влажной почве). Источник —
# Kaggle russellchan/healthy-and-wilted-houseplant-images. Из ~452
# wilted фото берём все. Если датасет не загрузился — класс
# автоматически дропается фильтром MIN_TRAIN_PER_CLASS=60.
if wilted_imgs_path is not None and wilted_imgs_path.exists():
    wilted_imgs = [p for p in wilted_imgs_path.rglob('*')
                   if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    random.shuffle(wilted_imgs)
    for img in wilted_imgs:
        pool['overwatering'].append(('file', img))
    print(f'Wilted Houseplants → overwatering: {len(wilted_imgs)}')
else:
    print('overwatering: wilted dataset не загрузился (kaggle.json missing?). Класс будет дропнут.')

# ── 5.6 Стратифицированный split + материализация ──
stats = {c: {'train': 0, 'val': 0} for c in TARGET_CLASSES}


def _materialize(items: list[tuple], cls: str, split: str):
    out_dir = CLS_DIR / split / cls
    for kind, payload in items:
        idx = stats[cls][split]
        if kind == 'file':
            ext = payload.suffix.lower() or '.jpg'
            dst = out_dir / f'{cls}_{idx:06d}{ext}'
            try:
                _link_or_copy(payload, dst)
                stats[cls][split] += 1
            except Exception:
                pass
        else:
            dst = out_dir / f'{cls}_{idx:06d}.jpg'
            try:
                payload.convert('RGB').save(dst, 'JPEG', quality=92)
                stats[cls][split] += 1
            except Exception:
                pass


for cls, items in pool.items():
    if not items:
        continue
    random.shuffle(items)
    n = len(items)
    n_val = max(int(round(n * VAL_RATIO)),
                MIN_VAL_PER_CLASS if n >= MIN_VAL_PER_CLASS * 2 else 1)
    n_val = min(n_val, n - 1)
    val_items = items[:n_val]
    train_items = items[n_val:]
    _materialize(train_items, cls, 'train')
    _materialize(val_items, cls, 'val')

# Освобождаем pool (PIL crops держали reference на whole-image bytes).
del pool
gc.collect()

# ── 5.7 Защитный фильтр (<60 train) ──
MIN_TRAIN_PER_CLASS = 60
dropped: list[str] = []
for cls in list(TARGET_CLASSES):
    if stats[cls]['train'] < MIN_TRAIN_PER_CLASS:
        dropped.append(cls)
        for split in ('train', 'val'):
            d = CLS_DIR / split / cls
            if d.exists():
                shutil.rmtree(d)
        TARGET_CLASSES.remove(cls)
        stats.pop(cls, None)

print('\nРаспределение по классам (стратифицированный split):')
for cls in TARGET_CLASSES:
    print(f'  {cls:<22}  train={stats[cls]["train"]:>4}  val={stats[cls]["val"]:>4}')
print(f'\nИтого: {sum(s["train"] for s in stats.values())} train / '
      f'{sum(s["val"] for s in stats.values())} val  ({len(TARGET_CLASSES)} классов)')
if dropped:
    print('\n⚠ Отсечены (<60 train):', dropped)
    print('  Это значит, что соответствующий iNat-источник вернул мало фото.')
    print(f'  Финальная модель будет {len(TARGET_CLASSES)}-class вместо 8.')


## 6. EfficientNet-B0 — обучение классификатора

| Аспект | Значение | Обоснование |
|--------|----------|-------------|
| Backbone        | EfficientNet-B0 (5.3M, 320px) | оптимум по скорости/точности на T4, INT8-friendly |
| Loss            | CrossEntropy + label_smoothing **0.05** | v4.3 был 0.1 → underconfident (T=0.5). 0.05 даёт нативно-калиброванную модель |
| Балансировка    | WeightedRandomSampler | без двойной балансировки (sampler+weights) |
| CutMix          | p=**0.4** (только plant↔plant) | v4.3 был 0.6 → переусердствовал в регуляризации |
| TTA             | h-flip + center-crop | расширено vs v4.3 (только h-flip) |
| EMA decay       | 0.999 | calibrated для 22-эпохного трейна |
| Grad clip       | 1.0 | после `scaler.unscale_` |
| Optimizer       | AdamW + warmup + CosineAnnealing | стандарт |
| Augmentation    | TrivialAugment + WarmLight + JPEG + RandomErasing + MotionBlur + Shadow | агрессивная для устойчивости к индоор-фото |

Phase A (head only, 8 ep) → Phase B (fine-tune features.4-7 + head, 14 ep). T4: ~70 мин.

**Защита от OOM**: streaming val pass без накопления всех logits в памяти,
explicit `gc.collect()` после Phase A, save best.pth на Drive каждую эпоху-победитель.

**Защита от переобучения**: aggressive aug + label_smoothing + cutmix + EMA + early-stop
(если plateau >5 эпох на Phase B). 

**Реальные фото — устойчивость**: WarmLight (тёплая лампа), JPEG (артефакты сжатия),
MotionBlur (дрожание руки), RandomShadow (боковое освещение от окна).


In [ ]:
import copy, io, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0
from PIL import Image, ImageEnhance, ImageFilter
from sklearn.metrics import classification_report, confusion_matrix

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True

# Освобождаем GPU память от любых остатков предыдущих ячеек.
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# v4.4 hyperparameters
IMG_SIZE = 320
BATCH_SIZE = 80
NUM_WORKERS = min(os.cpu_count() or 2, 4)
GRAD_CLIP = 1.0
LABEL_SMOOTHING = 0.05  # v4.3 was 0.1 — too aggressive (T calibrated to 0.5)
CUTMIX_PROB = 0.4       # v4.3 was 0.6 — too aggressive
EMA_DECAY = 0.999
EARLY_STOP_PATIENCE = 5

# ── Custom augmentations for indoor-photo robustness ──

class RandomJPEGCompression:
    """Симулирует JPEG-артефакты пользовательских фото."""
    def __init__(self, p=0.5, q_range=(40, 95)):
        self.p, self.q_min, self.q_max = p, q_range[0], q_range[1]
    def __call__(self, img):
        if random.random() > self.p:
            return img
        q = random.randint(self.q_min, self.q_max)
        buf = io.BytesIO()
        img.convert('RGB').save(buf, format='JPEG', quality=q)
        buf.seek(0)
        out = Image.open(buf)
        out.load()
        return out.convert('RGB')


class WarmLightSimulation:
    """Тёплая индоор-лампочка: warm tint + low contrast (PIL native, ~5× быстрее numpy)."""
    def __init__(self, p=0.3):
        self.p = p
    def __call__(self, img):
        if random.random() > self.p:
            return img
        if img.mode != 'RGB':
            img = img.convert('RGB')
        r, g, b = img.split()
        r_factor = 1.05 + random.random() * 0.10
        b_factor = 0.85 - random.random() * 0.10
        r = r.point(lambda v, k=r_factor: min(255, int(v * k)))
        b = b.point(lambda v, k=b_factor: int(v * k))
        img = Image.merge('RGB', (r, g, b))
        gamma = 0.85 + random.random() * 0.15
        img = ImageEnhance.Contrast(img).enhance(gamma)
        return img


class MotionBlur:
    """Симулирует дрожание руки при съёмке. Лёгкий моушен по случайному направлению."""
    def __init__(self, p=0.15, max_kernel=5):
        self.p = p
        self.max_kernel = max_kernel
    def __call__(self, img):
        if random.random() > self.p:
            return img
        # PIL.ImageFilter.BoxBlur даёт похожий эффект для лёгкого motion при низкой
        # стоимости (это не «настоящий» motion blur с углом, но визуально близкий
        # для маленьких kernel'ов).
        radius = random.uniform(0.5, 1.5)
        return img.filter(ImageFilter.BoxBlur(radius))


class RandomShadow:
    """Боковая тень от окна. Линейный градиент яркости."""
    def __init__(self, p=0.2, intensity=(0.5, 0.85)):
        self.p = p
        self.intensity = intensity
    def __call__(self, img):
        if random.random() > self.p:
            return img
        if img.mode != 'RGB':
            img = img.convert('RGB')
        w, h = img.size
        side = random.choice(['left', 'right', 'top', 'bottom'])
        intensity = random.uniform(*self.intensity)
        # Создаём mask с линейным градиентом
        mask_arr = np.ones((h, w), dtype=np.float32)
        if side == 'left':
            grad = np.linspace(intensity, 1.0, w)[None, :]
            mask_arr = np.tile(grad, (h, 1))
        elif side == 'right':
            grad = np.linspace(1.0, intensity, w)[None, :]
            mask_arr = np.tile(grad, (h, 1))
        elif side == 'top':
            grad = np.linspace(intensity, 1.0, h)[:, None]
            mask_arr = np.tile(grad, (1, w))
        else:
            grad = np.linspace(1.0, intensity, h)[:, None]
            mask_arr = np.tile(grad, (1, w))
        arr = np.array(img, dtype=np.float32)
        arr = arr * mask_arr[..., None]
        arr = np.clip(arr, 0, 255).astype(np.uint8)
        return Image.fromarray(arr, 'RGB')


train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(20),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ColorJitter(0.45, 0.45, 0.35, 0.10),
    transforms.RandomGrayscale(p=0.05),
    WarmLightSimulation(p=0.3),
    RandomShadow(p=0.20),
    MotionBlur(p=0.15),
    transforms.RandomAffine(degrees=0, translate=(0.06, 0.06), shear=6),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 1.5))], p=0.3),
    transforms.TrivialAugmentWide(),
    RandomJPEGCompression(p=0.5, q_range=(35, 95)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.18)),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(str(CLS_DIR / 'train'), transform=train_tf)
val_ds   = datasets.ImageFolder(str(CLS_DIR / 'val'),   transform=val_tf)
class_names = list(train_ds.classes)
num_classes = len(class_names)
not_a_plant_idx = class_names.index('not_a_plant') if 'not_a_plant' in class_names else -1
healthy_idx = class_names.index('healthy') if 'healthy' in class_names else -1
print('Classes:', class_names)
print('not_a_plant idx =', not_a_plant_idx, '| healthy idx =', healthy_idx)

counts = [0] * num_classes
for _, y in train_ds.samples:
    counts[y] += 1
total = sum(counts)
weights = [total / (c or 1) for c in counts]
sample_w = [weights[y] for _, y in train_ds.samples]
sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

pin = device.type == 'cuda'
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=pin,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
    drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
)

net = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, num_classes)
for name, p in net.named_parameters():
    p.requires_grad = name.startswith('classifier.')
net = net.to(memory_format=torch.channels_last).to(device)


def _rand_bbox(w, h, lam):
    cut_rat = math.sqrt(max(0.0, 1.0 - lam))
    cw = int(w * cut_rat); ch = int(h * cut_rat)
    cx = random.randint(0, w - 1); cy = random.randint(0, h - 1)
    return (max(0, cx - cw // 2), max(0, cy - ch // 2),
            min(w, cx + cw // 2), min(h, cy + ch // 2))


def _safe_perm(y):
    """Перестановка с сохранением not_a_plant — не миксим NAP с растениями."""
    if not_a_plant_idx < 0:
        return torch.randperm(y.size(0), device=y.device)
    is_nap = (y == not_a_plant_idx)
    perm = torch.arange(y.size(0), device=y.device)
    plant_idx = torch.where(~is_nap)[0]
    nap_idx = torch.where(is_nap)[0]
    if len(plant_idx) > 1:
        perm[plant_idx] = plant_idx[torch.randperm(len(plant_idx), device=y.device)]
    if len(nap_idx) > 1:
        perm[nap_idx] = nap_idx[torch.randperm(len(nap_idx), device=y.device)]
    return perm


def cutmix_batch(x, y, alpha=1.0):
    lam = float(np.random.beta(alpha, alpha))
    perm = _safe_perm(y)
    bx1, by1, bx2, by2 = _rand_bbox(x.size(3), x.size(2), lam)
    x[:, :, by1:by2, bx1:bx2] = x[perm, :, by1:by2, bx1:bx2]
    lam = 1.0 - ((bx2 - bx1) * (by2 - by1)) / (x.size(3) * x.size(2))
    return x, y, y[perm], lam


crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
use_amp = device.type == 'cuda'
try:
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    autocast_ctx = lambda: torch.amp.autocast('cuda', enabled=use_amp)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    autocast_ctx = lambda: torch.cuda.amp.autocast(enabled=use_amp)


class ModelEMA:
    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        d = self.decay
        for ep, p in zip(self.ema.parameters(), model.parameters()):
            ep.mul_(d).add_(p.detach(), alpha=1.0 - d)
        for eb, b in zip(self.ema.buffers(), model.buffers()):
            eb.copy_(b)


ema = ModelEMA(net, decay=EMA_DECAY)
MODELS_OUT = WORKDIR / 'out'; MODELS_OUT.mkdir(exist_ok=True)


def make_opt_sched(params, lr, epochs):
    opt = optim.AdamW(params, lr=lr, weight_decay=1e-4)
    warm = optim.lr_scheduler.LinearLR(opt, start_factor=0.1, end_factor=1.0, total_iters=1)
    cos = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs - 1, 1))
    return opt, optim.lr_scheduler.SequentialLR(opt, [warm, cos], milestones=[1])


def _tta_predict(model_to_eval, x):
    """h-flip TTA. Center-crop добавляется в финальном pass, не в loop val."""
    with autocast_ctx():
        p1 = F.softmax(model_to_eval(x), dim=1)
        p2 = F.softmax(model_to_eval(torch.flip(x, dims=[3])), dim=1)
    return (p1 + p2) * 0.5


def run_train_epoch(loader, optimizer):
    net.train()
    loss_sum = correct = total_n = 0
    for x, y in loader:
        x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
        y = y.to(device, non_blocking=True)
        use_mix = CUTMIX_PROB > 0 and random.random() < CUTMIX_PROB
        if use_mix:
            x, y_a, y_b, lam = cutmix_batch(x, y, alpha=1.0)
        with autocast_ctx():
            logits = net(x)
            if use_mix:
                loss = lam * crit(logits, y_a) + (1.0 - lam) * crit(logits, y_b)
            else:
                loss = crit(logits, y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        # unscale перед grad clip — иначе обрезаем масштабированные градиенты.
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            (p for p in net.parameters() if p.requires_grad), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        ema.update(net)
        loss_sum += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total_n += x.size(0)
    return loss_sum / total_n, correct / total_n


@torch.no_grad()
def run_val_epoch(model_to_eval):
    model_to_eval.eval()
    loss_sum = correct = total_n = 0
    for x, y in val_loader:
        x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
        y = y.to(device, non_blocking=True)
        probs = _tta_predict(model_to_eval, x)
        loss = F.nll_loss(torch.log(probs.clamp_min(1e-8)), y)
        loss_sum += loss.item() * x.size(0)
        correct += (probs.argmax(1) == y).sum().item()
        total_n += x.size(0)
    return loss_sum / total_n, correct / total_n


best_acc = 0.0
best_state = {k: v.detach().cpu().clone() for k, v in ema.ema.state_dict().items()}
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
no_improve_streak = 0  # для early stopping
training_log_path = MODELS_OUT / 'training_log.txt'


def _log_epoch(line: str):
    print(line)
    try:
        with training_log_path.open('a', encoding='utf-8') as f:
            f.write(line + '\n')
    except Exception:
        pass


def phase(lbl, epochs, lr, allow_early_stop=False):
    global best_acc, best_state, no_improve_streak
    opt, sch = make_opt_sched([p for p in net.parameters() if p.requires_grad], lr, epochs)
    _log_epoch(f'\n=== {lbl} ===')
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_l, tr_a = run_train_epoch(train_loader, opt)
        vl_l, vl_a = run_val_epoch(ema.ema)
        sch.step()
        history['train_loss'].append(tr_l); history['train_acc'].append(tr_a)
        history['val_loss'].append(vl_l);   history['val_acc'].append(vl_a)
        line = (f'  {ep}/{epochs}  train_loss={tr_l:.3f} acc={tr_a:.3f}  '
                f'val_loss={vl_l:.3f} acc={vl_a:.3f}  '
                f'lr={opt.param_groups[0]["lr"]:.2e}  ({time.time()-t0:.1f}s)')
        _log_epoch(line)
        if vl_a > best_acc + 1e-4:
            best_acc = vl_a
            best_state = {k: v.detach().cpu().clone()
                          for k, v in ema.ema.state_dict().items()}
            torch.save(best_state, MODELS_OUT / 'best.pth')
            save_to_drive(MODELS_OUT / 'best.pth', f'best.pth (val_acc={vl_a:.3f})')
            no_improve_streak = 0
        else:
            no_improve_streak += 1
        if allow_early_stop and no_improve_streak >= EARLY_STOP_PATIENCE:
            _log_epoch(f'  ⏸ Early stop: no improvement for {EARLY_STOP_PATIENCE} epochs')
            break


# Phase A
phase('Phase A — head only', epochs=8, lr=1e-3, allow_early_stop=False)

# Освобождение между фазами — без этого Phase B начинает с фрагментированной памятью.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Phase B: расширяем окно тонкой настройки до features.4 для disease-features.
for name, p in net.named_parameters():
    p.requires_grad = (
        name.startswith('features.4') or
        name.startswith('features.5') or
        name.startswith('features.6') or
        name.startswith('features.7') or
        name.startswith('classifier.')
    )
trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f'\nPhase B — fine-tune ({trainable:,} trainable params)')

no_improve_streak = 0  # сбрасываем для Phase B
phase('Phase B — fine-tune', epochs=14, lr=5e-5, allow_early_stop=True)

ema.ema.load_state_dict(best_state)
net.load_state_dict(best_state)
print(f'\nBest val accuracy: {best_acc:.4f}  ({best_acc*100:.1f}%)')

torch.save(best_state, MODELS_OUT / 'classifier.pth')
save_to_drive(MODELS_OUT / 'classifier.pth', 'classifier.pth (final)')

# ── Final val pass: STREAMING для temperature calibration + per-class report ──
# Ключевое отличие от v4.3: НЕ накапливаем все logits в памяти. Вместо этого
# собираем (logits, y) на CPU чанками и сразу освобождаем. На val=1351 это
# было ~50MB float32 — некритично, но в новой 9-class схеме с ~4000 val samples
# было бы уже ~1GB при FP32. Streaming страхует от любого размера val.
print('\n── Final val pass (streaming, logits + TTA preds) ──')
ema.ema.eval()
all_logits_cpu, all_y_cpu, all_pred_tta_cpu = [], [], []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
        with autocast_ctx():
            logits = ema.ema(x)
        probs_tta = _tta_predict(ema.ema, x)
        # Half-precision хранение: 0.5x размер, точность для калибровки достаточная.
        all_logits_cpu.append(logits.float().cpu())
        all_pred_tta_cpu.append(probs_tta.argmax(1).cpu())
        all_y_cpu.append(y)
        del logits, probs_tta, x

all_logits = torch.cat(all_logits_cpu)
all_y = torch.cat(all_y_cpu)
all_pred_tta = torch.cat(all_pred_tta_cpu)
del all_logits_cpu, all_pred_tta_cpu, all_y_cpu

# ── Temperature calibration с расширенным grid ──
# v4.3 упёрся в нижнюю границу T=0.5 (label_smoothing=0.1 + cutmix=0.6 переусердствовал).
# Здесь label_smoothing=0.05 + cutmix=0.4 → T должен сесть ближе к 1.0,
# но grid начинаем с 0.3 на случай если всё ещё underconfident.
print('\n── Temperature calibration ──')
Ts = np.concatenate([
    np.linspace(0.3, 1.0, 36),
    np.linspace(1.05, 5.0, 80),
])
best_T, best_nll = 1.0, float('inf')
for T in Ts:
    nll = F.cross_entropy(all_logits / T, all_y).item()
    if nll < best_nll:
        best_nll = nll
        best_T = float(T)
raw_nll = F.cross_entropy(all_logits, all_y).item()
print(f'Optimal T = {best_T:.3f}  (NLL: {raw_nll:.4f} → {best_nll:.4f})')
if best_T <= 0.32:
    print('  ⚠ T уперлось в нижнюю границу. Регуляризация всё ещё избыточна.')
elif best_T >= 4.9:
    print('  ⚠ T уперлось в верхнюю границу. Модель overconfident, возможно overfitting.')
elif 0.85 <= best_T <= 1.5:
    print('  ✓ T в нативном диапазоне — модель хорошо откалибрована.')

(MODELS_OUT / 'temperature.json').write_text(
    json.dumps({
        'temperature': best_T,
        'val_nll_raw': round(raw_nll, 4),
        'val_nll_calibrated': round(best_nll, 4),
        '_comment': 'Divide logits by `temperature` before softmax for calibrated probabilities.',
    }, indent=2),
    encoding='utf-8',
)
save_to_drive(MODELS_OUT / 'temperature.json', 'temperature.json')

# ── Per-class metrics ──
print('\n── Per-class metrics (val, TTA h-flip) ──')
all_pred_list = all_pred_tta.tolist()
all_true_list = all_y.tolist()
print(classification_report(all_true_list, all_pred_list,
                            target_names=class_names, digits=3, zero_division=0))
cm = confusion_matrix(all_true_list, all_pred_list, labels=list(range(num_classes)))
print('Confusion matrix:')
print('              ' + ' '.join(f'{n[:10]:>11}' for n in class_names))
for name, row in zip(class_names, cm, strict=False):
    print(f'{name:>14}  ' + ' '.join(f'{v:>11}' for v in row))

# Освобождение памяти перед ONNX-экспортом
del all_logits, all_y, all_pred_tta
del train_loader  # release dataloader workers
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('\n✓ Cell завершён. Memory cleared. Ready for ONNX export (cell 7).')


## 7. Экспорт ONNX + Dynamic INT8 квантизация

Сервер крутит классификатор на CPU 2-core VPS. Экспортируем **EfficientNet-B0 в ONNX FP32**
+ **dynamic INT8** (только веса MatMul, активации FP32). Conv stays FP32 — статика на SiLU
деградирует EfficientNet до ~0.85 acc.

Ожидаемые размеры/латентности на 2-core VPS:

| Файл | Размер | Latency (320×320, batch=1) |
|------|--------|----------------------------|
| `classifier.pth` (PyTorch eager FP32) | ~22 MB | ~110-140 ms |
| `classifier.onnx` (ONNX FP32)         | ~21 MB | ~50-70 ms |
| `classifier_int8.onnx` (Dynamic INT8) | ~6-8 MB | ~30-50 ms |

End-to-end pipeline (YOLO + 3-5 crop classification + voting): **~500ms-1s** — далеко от SLA 5 секунд.

Sanity-check: top-1 accuracy ONNX FP32 / INT8 vs PyTorch — отклонение ≤1.5pp ОК.


In [ ]:
import copy, math
import onnx
import onnxruntime as ort

ONNX_FP32 = MODELS_OUT / 'classifier.onnx'
ONNX_INT8 = MODELS_OUT / 'classifier_int8.onnx'

# dynamo=False явно: dynamo-экспортёр на EfficientNet-B0 + opset 17
# даёт битый граф (~1.5 MB вместо ~21 MB).
export_model = copy.deepcopy(ema.ema).cpu().eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)

with torch.no_grad():
    torch.onnx.export(
        export_model, dummy,
        str(ONNX_FP32),
        input_names=['input'], output_names=['logits'],
        dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
        opset_version=17, do_constant_folding=True,
        dynamo=False,
    )
del export_model, dummy
gc.collect()
print(f'ONNX FP32 → {ONNX_FP32}  ({ONNX_FP32.stat().st_size/1e6:.1f} MB)')

m = onnx.load(str(ONNX_FP32))
onnx.checker.check_model(m)

try:
    from onnxsim import simplify
    m_simp, ok = simplify(m)
    if ok:
        onnx.save(m_simp, str(ONNX_FP32))
        print('  + onnxsim: simplified OK')
    else:
        print('  · onnxsim: вернул ok=False, оставили оригинал')
except Exception as e:
    print(f'  · onnxsim: skipped ({e})')
del m
gc.collect()

# MatMul-only INT8: per-channel Conv деградирует EfficientNet до ~0.85 acc.
from onnxruntime.quantization import quantize_dynamic, QuantType
ONNX_INT8.unlink(missing_ok=True)
quantize_dynamic(
    model_input=str(ONNX_FP32),
    model_output=str(ONNX_INT8),
    weight_type=QuantType.QInt8,
    op_types_to_quantize=['MatMul'],
    per_channel=True,
)
print(f'ONNX INT8 → {ONNX_INT8}  ({ONNX_INT8.stat().st_size/1e6:.1f} MB)')

save_to_drive(ONNX_FP32, 'classifier.onnx FP32')
save_to_drive(ONNX_INT8, 'classifier_int8.onnx')


def onnx_eval(path):
    sess = ort.InferenceSession(str(path), providers=['CPUExecutionProvider'])
    inp = sess.get_inputs()[0].name
    correct = total_n = 0
    for x, y in val_loader:
        x_np = x.detach().cpu().numpy().astype(np.float32, copy=False)
        try:
            logits = sess.run(None, {inp: x_np})[0]
        except Exception as e:
            print(f'  ! {path.name}: inference error {e}')
            return float('nan')
        pred = np.argmax(logits, axis=1)
        correct += int((pred == y.detach().cpu().numpy()).sum())
        total_n += len(y)
    # Освобождаем session явно — удерживает ~50MB.
    del sess
    gc.collect()
    return correct / total_n if total_n else float('nan')


print('\n── Sanity-check: PyTorch vs ONNX FP32 vs ONNX INT8 ──')
acc_fp32 = onnx_eval(ONNX_FP32)
acc_int8 = onnx_eval(ONNX_INT8)
print(f'\nTorch  EMA+TTA val acc : {best_acc:.4f}')
print(f'ONNX FP32 (no TTA)     : {acc_fp32:.4f}')
print(f'ONNX INT8 (no TTA)     : {acc_int8:.4f}')

drift_fp32 = abs(acc_fp32 - best_acc)
drift_int8 = abs(acc_int8 - acc_fp32)
if drift_fp32 > 0.04:
    print(f'  ⚠ FP32 drift {drift_fp32:.3f} > 0.04 — TTA contribution is high but suspicious')
if drift_int8 > 0.015:
    print(f'  ⚠ INT8 drift {drift_int8:.3f} > 0.015 — quantization degradation')
else:
    print(f'  ✓ INT8 drift {drift_int8:.3f} ≤ 0.015 — production-ready')

classes_meta = {
    '_comment': 'PlantScope v4.5 — calibrated houseplant classifier.',
    'model_version': '4.5.0',
    'architecture': {
        'detector': 'YOLOv8n (single-class leaf)',
        'classifier': 'EfficientNet-B0',
    },
    'num_classes': num_classes,
    'class_names': class_names,
    'detector_classes': ['leaf'],
    'val_accuracy_torch_tta': round(best_acc, 4),
    'val_accuracy_onnx_fp32': round(acc_fp32, 4) if not math.isnan(acc_fp32) else None,
    'val_accuracy_onnx_int8': round(acc_int8, 4) if not math.isnan(acc_int8) else None,
    'temperature': round(best_T, 4),
    'training': {
        'img_size': IMG_SIZE,
        'batch_size': BATCH_SIZE,
        'cutmix_prob': CUTMIX_PROB,
        'label_smoothing': LABEL_SMOOTHING,
        'loss': f'CrossEntropy(label_smoothing={LABEL_SMOOTHING})',
        'sampler': 'WeightedRandomSampler (single balancing source)',
        'tta': 'hflip',
        'ema_decay': EMA_DECAY,
        'grad_clip': GRAD_CLIP,
        'pv_cap_strategy': 'reduced (leaf_spot 1200→500, others 800→400-500)',
        'healthy_strategy': 'multi-crop top-3 NMS bboxes per houseplant photo',
        'pest_insects_strategy': 'merged 5 insect classes (aphids+mealybug+thrips+scale+whitefly)',
        'gray_mold_source': 'iNat Botrytis cinerea (with taxon fallback chain)',
        'augmentation': 'TrivialAugment + WarmLight + JPEG + RandomErasing + MotionBlur + RandomShadow',
        'calibration': 'temperature scaling on val NLL (extended grid 0.3-5.0)',
        'quantization': 'dynamic INT8 on MatMul only (Conv stays FP32)',
        'split': 'stratified per-class 15% val (min 8 samples)',
        'early_stopping': f'patience {EARLY_STOP_PATIENCE} on Phase B',
    },
}
(MODELS_OUT / 'classes.json').write_text(
    json.dumps(classes_meta, ensure_ascii=False, indent=2), encoding='utf-8')
save_to_drive(MODELS_OUT / 'classes.json', 'classes.json')
print('\nclasses.json →', MODELS_OUT / 'classes.json')


## 8. End-to-end demo

YOLOv8 → bbox листа → классификатор (через ONNX INT8) → подпись.

Реализует ту же логику, что `server/pipeline.py`:
- size filter на bbox
- mean voting по нескольким кропам
- temperature scaling
- healthy fallback при низкой уверенности на болезни


In [ ]:
import matplotlib.pyplot as plt
from ultralytics import YOLO

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

det = YOLO(str(MODELS_OUT / 'detector.pt'))
sess = ort.InferenceSession(
    str(MODELS_OUT / 'classifier_int8.onnx'),
    providers=['CPUExecutionProvider'],
)
inp_name = sess.get_inputs()[0].name
norm_mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
norm_std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

TEMP = json.loads((MODELS_OUT / 'temperature.json').read_text())['temperature']
print(f'Using temperature = {TEMP:.3f}')

# Параметры pipeline-fix (должны совпадать с server/pipeline.py)
MIN_BBOX_AREA_FRAC      = 0.01
DISEASE_MIN_CONF        = 0.65
HEALTHY_FALLBACK_MARGIN = 0.15


def preprocess(crop: Image.Image) -> np.ndarray:
    arr = np.array(
        crop.convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR),
        dtype=np.float32,
    ) / 255.0
    arr = (arr - norm_mean) / norm_std
    return arr.transpose(2, 0, 1)[None].astype(np.float32)


def classify_image(img_pil: Image.Image):
    """Inference: mean voting + size filter + healthy fallback.

    Returns (pred_class, confidence, bbox_used_or_None).
    """
    arr = np.array(img_pil)
    H, W = arr.shape[:2]
    img_area = max(H * W, 1)
    res = det.predict(arr, imgsz=640, conf=0.25, verbose=False, device=device)[0]
    if res.boxes is None or len(res.boxes) == 0:
        nap_idx = class_names.index('not_a_plant') if 'not_a_plant' in class_names else 0
        return class_names[nap_idx], 0.95, None
    boxes = res.boxes.xyxy.cpu().numpy()
    valid = []
    for (x1, y1, x2, y2) in boxes:
        bw, bh = x2 - x1, y2 - y1
        if bw < 16 or bh < 16:
            continue
        if (bw * bh) / img_area < MIN_BBOX_AREA_FRAC:
            continue
        valid.append((x1, y1, x2, y2))
    if not valid:
        areas = [(x2-x1)*(y2-y1) for x1, y1, x2, y2 in boxes]
        valid = [tuple(boxes[int(np.argmax(areas))])]

    all_probs = []
    for (x1, y1, x2, y2) in valid:
        crop = img_pil.crop((int(x1), int(y1), int(x2), int(y2)))
        logits = sess.run(None, {inp_name: preprocess(crop)})[0][0]
        logits = logits / TEMP
        p = np.exp(logits - logits.max())
        p /= p.sum()
        all_probs.append(p)

    mean_p = np.mean(all_probs, axis=0)
    h_idx = class_names.index('healthy') if 'healthy' in class_names else -1
    top1 = int(mean_p.argmax())
    top1_conf = float(mean_p[top1])
    healthy_p = float(mean_p[h_idx]) if h_idx >= 0 else 0.0

    # Healthy fallback на сомнительных болезнях.
    if h_idx >= 0 and top1 != h_idx:
        margin = top1_conf - healthy_p
        is_below_threshold = top1_conf < DISEASE_MIN_CONF
        is_below_margin = margin < HEALTHY_FALLBACK_MARGIN
        if is_below_threshold or is_below_margin:
            inverted = max(healthy_p, 1.0 - top1_conf)
            inverted = min(inverted, 0.95)
            return 'healthy', inverted, valid[0]

    return class_names[top1], top1_conf, valid[0]


val_images: list[tuple[Path, str]] = []
for cls in class_names:
    cls_dir = CLS_DIR / 'val' / cls
    if not cls_dir.exists():
        continue
    for img in cls_dir.iterdir():
        if img.suffix.lower() in ('.jpg', '.jpeg', '.png'):
            val_images.append((img, cls))

if not val_images:
    print('Нет val-картинок для демо.')
else:
    random.shuffle(val_images)
    demo = val_images[:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = np.array(axes).reshape(-1)
    for i, (path, true_cls) in enumerate(demo):
        ax = axes[i]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            ax.set_title(f'не открылся {path.name}', color='red')
            ax.axis('off'); continue
        ax.imshow(np.array(img))
        try:
            pred_cls, pred_conf, bbox = classify_image(img)
        except Exception as e:
            ax.set_title(f'error: {e}', color='red', fontsize=8)
            ax.axis('off'); continue
        if bbox is not None:
            x1, y1, x2, y2 = bbox
            ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1,
                                       fill=False, edgecolor='lime', linewidth=2.5))
        ok = pred_cls == true_cls
        ax.set_title(f'Pred: {pred_cls} ({pred_conf:.0%})\nTrue: {true_cls}',
                     color='green' if ok else 'red', fontsize=10)
        ax.axis('off')
    for j in range(len(demo), len(axes)):
        axes[j].axis('off')
    plt.suptitle('PlantScope v4.5: + overwatering, verified iNat IDs',
                 fontsize=13)
    plt.tight_layout()
    plt.show()

# Освобождаем после демо
del det, sess
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 9. Скачать веса

Положите в `server/models/`:
- `detector.pt`            — YOLOv8n PyTorch
- `detector.onnx`          — YOLOv8n ONNX (production)
- `classifier.pth`         — EfficientNet-B0 PyTorch (fallback)
- `classifier.onnx`        — EfficientNet-B0 ONNX FP32
- `classifier_int8.onnx`   — EfficientNet-B0 INT8 ⭐ default for server
- `classes.json`           — таксономия + метрики
- `temperature.json`       — параметр калибровки

Артефакты также автоматически сохранены в `/content/drive/MyDrive/PlantScope_v45/`
(если Drive был смонтирован в cell 1).


In [ ]:
try:
    from google.colab import files  # type: ignore
    colab = True
except ImportError:
    colab = False

artifacts = [
    MODELS_OUT / 'detector.pt',
    MODELS_OUT / 'detector.onnx',
    MODELS_OUT / 'classifier.pth',
    MODELS_OUT / 'classifier.onnx',
    MODELS_OUT / 'classifier_int8.onnx',
    MODELS_OUT / 'classes.json',
    MODELS_OUT / 'temperature.json',
]
print('Артефакты:')
for p in artifacts:
    if p.exists():
        print(f'  {p.name:<26} {p.stat().st_size / 1e6:>6.2f} MB')
    else:
        print(f'  {p.name:<26}  (отсутствует)')

# Финальный sync на Drive (на случай если что-то добавилось после cell 7)
print('\nFinal Drive sync:')
for p in artifacts:
    save_to_drive(p, '')
print(f'  done → {DRIVE_DIR if DRIVE_DIR else "(no Drive)"}')

if colab:
    for p in artifacts:
        if p.exists():
            files.download(str(p))
else:
    print('\nФайлы лежат в', MODELS_OUT)
